# 06 — Local Chatbot
Chatbot RAG menggunakan Ollama (Phi-3 Mini) + Chroma local.
Semua berjalan 100% offline — tidak ada API call ke luar!

**Stack:**
- LLM    : Ollama + phi3:mini (local)
- Vector : Chroma + gte-large (local)
- RAG    : query → retrieve → generate

## 0. Cek Ollama
Pastikan Ollama sudah running dan model phi3:mini tersedia.

In [1]:
import requests

try:
    response = requests.get("http://localhost:11434/api/tags")
    models = [m['name'] for m in response.json()['models']]
    print("Ollama running!")
    print(f"Model tersedia:")
    for m in models:
        print(f"  - {m}")
except:
    print("Ollama tidak running! Jalankan Ollama dulu.")

Ollama running!
Model tersedia:
  - phi3:mini


## 1. Load Chroma + Embedding Model
Load vector store dan embedding model dari disk.

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-large"
)

vectorstore = Chroma(
    collection_name="phis_sds",
    embedding_function=embeddings,
    persist_directory="../vector_db"
)

print(f"Vector store loaded!")
print(f"Total vectors : {vectorstore._collection.count()}")

C:\Users\Dev\AppData\Local\Temp\ipykernel_12448\69707153.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\Dev\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2315.86it/s]
C:\Users\Dev\AppData\Local\Temp\ipykernel_12448\69707153.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class e

Vector store loaded!
Total vectors : 1125


## 2. Setup RAG + Ollama
Buat fungsi yang menggabungkan Chroma retrieval dengan Ollama LLM.

In [5]:
import requests
import json

def chat(query, n_results=3):
    # 1. Retrieve chunks dari Chroma
    docs = vectorstore.similarity_search(query, k=n_results)
    
    # 2. Gabungkan chunks jadi context
    context = "\n\n---\n\n".join([
        f"[Page {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for doc in docs
    ])
    
    # 3. Buat prompt
    prompt = f"""You are a helpful assistant for PHIS (Pharmacy Information System) documentation.
Answer the question based ONLY on the context provided below.
If the answer is not in the context, say "I don't have information about that in the documentation."

Context:
{context}

Question: {query}

Answer:"""
    
    # 4. Kirim ke Ollama
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi3:mini",
            "prompt": prompt,
            "stream": False
        }
    )
    
    answer = response.json()['response']
    
    print(f"Question: {query}")
    print(f"{'='*55}")
    print(f"Answer: {answer}")
    print(f"{'='*55}")
    print(f"Sources:")
    for i, doc in enumerate(docs):
        print(f"  [{i+1}] Page {doc.metadata.get('page', 'N/A')}")

# Test pertama!
chat("who can approve SAM request at facility level?")

Question: who can approve SAM request at facility level?
Answer: The Hospital Director or District Health Officer has the provision to endorse (approve) SAM Requests at the facility level as per the context provided on Page 84 of PHIS documentation.
Sources:
  [1] Page 84
  [2] Page 41
  [3] Page 66


## 3. Chat dengan Memory
Tambahkan history percakapan supaya chatbot ingat konteks sebelumnya.

In [6]:
def chat_with_memory(query, history=[], n_results=3):
    # 1. Retrieve chunks
    docs = vectorstore.similarity_search(query, k=n_results)
    context = "\n\n---\n\n".join([
        f"[Page {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for doc in docs
    ])
    
    # 2. Format history
    history_text = ""
    if history:
        for h in history[-3:]:  # ambil 3 history terakhir
            history_text += f"User: {h['user']}\nAssistant: {h['assistant']}\n\n"
    
    # 3. Prompt dengan history
    prompt = f"""You are a helpful assistant for PHIS SAM documentation.
Answer based ONLY on the context provided.
If not found, say "I don't have information about that."

Previous conversation:
{history_text}

Context:
{context}

Question: {query}

Answer:"""
    
    # 4. Kirim ke Ollama
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "phi3:mini", "prompt": prompt, "stream": False}
    )
    answer = response.json()['response']
    
    # 5. Simpan ke history
    history.append({"user": query, "assistant": answer})
    
    print(f"User     : {query}")
    print(f"{'='*55}")
    print(f"Assistant: {answer}")
    print(f"{'='*55}\n")
    
    return history

# Test dengan 2 pertanyaan berurutan
history = []
history = chat_with_memory("what is SAM?", history)
history = chat_with_memory("what are its categories?", history)

User     : what is SAM?
Assistant: SAM stands for the System Design Specification, as mentioned in reference PhIS/SDS/SAM Page 179 on PHIS SAM documentation. It's a framework detailing how various datasets within an organization should be managed and maintained using the SAM system. The context indicates that it is used to manage dataset for multiple purposes including fund sourcing, monitoring categories, disclaimer-related details among others as specified in Appendix of its document reference on Page 190.

User     : what are its categories?
Assistant: The SAM (Specialized Accreditation Management) system has eight main data acquisition or "request" category types, which can be broadly categorized into three groups based on the nature of requestor roles. They are as follows:

1. Request Category A to H for Registered MAL users and their respective conditions (like MOH with different statuses). These categories also include non-MOH specialist requests from hospitals or clinics, along

## 4. Interactive Chat Loop
Chat interaktif — ketik pertanyaan langsung di notebook,
ketik 'exit' untuk keluar.

In [8]:
history = []

print("PHIS SAM Chatbot (phi3:mini + Chroma)")
print("Ketik 'exit' untuk keluar")
print("="*55)

while True:
    query = input("\nKamu: ")
    
    if query.lower() == "exit":
        print("Bye!")
        break
    
    if not query.strip():
        continue
    
    history = chat_with_memory(query, history)

PHIS SAM Chatbot (phi3:mini + Chroma)
Ketik 'exit' untuk keluar
User     : what is SAM approval process?
Assistant: The System for Award Management (SAM) Approval Process involves several steps, which are outlined in the documentation provided on PHIS pages. Initially, an approved Minimum Order Quantity (MOQ) request must be submitted by a Medical Officer Headquarters (MOH). Once received, it is forwarded to Section Heads for evaluation as per UI-PHIS-SAM-REQ SAM Request submission criteria outlined on Page 96.

If the Section Head finds any issues with the approval request and returns it back for revision or recommends endorsement of various aspects, this can be done through a 'Return to Revision' option (Page 66), whereby specific parts of the SAM Request are revised accordingly based on feedback.

After revisions have been made as required by Section Head(s) and other relevant stakeholders such as Endorsement, Deputy Director Health Officer Generalist/Hospital (DDGH/PS)/Deputy Direc